<a href="https://colab.research.google.com/github/ishwarikale44-byte/Practical-LP-3/blob/main/practical6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Step 1: Import libraries

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn import metrics


# Step 2: Read dataset

df = pd.read_csv("uber.csv")

print(df.head())
print(df.info())
print(df.describe())


# Step 3: Clean the dataset

df = df.drop(['Unnamed: 0', 'key'], axis=1)

df.dropna(inplace=True)

df['pickup_datetime'] = pd.to_datetime(
    df['pickup_datetime'], errors='coerce'
)

df.dropna(subset=['pickup_datetime'], inplace=True)


# Step 4: Extract date and time features

df['hour'] = df['pickup_datetime'].dt.hour
df['day'] = df['pickup_datetime'].dt.day
df['month'] = df['pickup_datetime'].dt.month
df['year'] = df['pickup_datetime'].dt.year

df.drop('pickup_datetime', axis=1, inplace=True)


# Step 5: Remove incorrect coordinates

df = df[
    (df['pickup_latitude'].between(-90, 90)) &
    (df['dropoff_latitude'].between(-90, 90)) &
    (df['pickup_longitude'].between(-180, 180)) &
    (df['dropoff_longitude'].between(-180, 180))
]


# Step 6: Calculate distance using Haversine formula

def distance_transform(lon1, lat1, lon2, lat2):

    lon1, lat1, lon2, lat2 = map(
        np.radians, [lon1, lat1, lon2, lat2]
    )

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = (
        np.sin(dlat / 2) ** 2 +
        np.cos(lat1) * np.cos(lat2) *
        np.sin(dlon / 2) ** 2
    )

    c = 2 * np.arcsin(np.sqrt(a))

    return 6371 * c


df['Distance'] = distance_transform(
    df['pickup_longitude'],
    df['pickup_latitude'],
    df['dropoff_longitude'],
    df['dropoff_latitude']
)


# Step 7: Identify and remove outliers

df = df[df['Distance'] < 60]
df = df[df['fare_amount'] > 0]

df = df[
    ~((df['fare_amount'] > 100) & (df['Distance'] < 1))
]


# Step 8: Visualize outliers

plt.figure(figsize=(8, 5))
sns.boxplot(x=df['fare_amount'])
plt.title("Outliers in Fare Amount")
plt.show()


# Step 9: Check correlation

corr = df.corr(numeric_only=True)

plt.figure(figsize=(10, 6))
sns.heatmap(corr, annot=True, cmap='BuGn')
plt.title("Correlation Heatmap")
plt.show()


# Step 10: Select input and output

X = df[['Distance']]
y = df['fare_amount']


# Step 11: Standardization

sc = StandardScaler()

X = sc.fit_transform(X)


# Step 12: Split dataset

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0
)


# Step 13: Linear Regression

lr = LinearRegression()

lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)


print("\nLinear Regression Results")

print("MAE:", metrics.mean_absolute_error(y_test, y_pred))

print("RMSE:", np.sqrt(
    metrics.mean_squared_error(y_test, y_pred)
))

print("R2 Score:", metrics.r2_score(y_test, y_pred))


# Step 14: Random Forest Regression

rf = RandomForestRegressor(
    n_estimators=100, random_state=10
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)


print("\nRandom Forest Regression Results")

print("MAE:", metrics.mean_absolute_error(y_test, y_pred_rf))

print("RMSE:", np.sqrt(
    metrics.mean_squared_error(y_test, y_pred_rf)
))

print("R2 Score:", metrics.r2_score(y_test, y_pred_rf))


# Step 15: Compare both models

result = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest'],
    'RMSE': [
        np.sqrt(metrics.mean_squared_error(y_test, y_pred)),
        np.sqrt(metrics.mean_squared_error(y_test, y_pred_rf))
    ],
    'R2 Score': [
        metrics.r2_score(y_test, y_pred),
        metrics.r2_score(y_test, y_pred_rf)
    ]
})

print("\nModel Comparison:")
print(result)


# Step 16: Visualization

plt.scatter(X_test, y_test, label="Actual")
plt.scatter(X_test, y_pred_rf, label="Predicted")

plt.xlabel("Standardized Distance")
plt.ylabel("Fare Amount")
plt.title("Actual vs Predicted Uber Fare")
plt.legend()
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'uber.csv'